# DMA 版 Sobel 验证（不需要显示器）

把 `test_720p.png` 通过 AXI DMA 送进 FPGA 上的 Sobel 核，读回来和黄金参考
`sobel_720p.png` 做**逐位**比对。

运行前确认本目录下有：`sobel_dma.bit`、`sobel_dma.hwh`、`sobel_dma.py`、
`test_720p.png`、`sobel_720p.png`。

说明见 `README.md`；核的设计与边界策略见仓库的 `src/hls/sobel/README.md`。

In [ ]:
import os, sys
import pynq
print('PYNQ', pynq.__version__)
print('cwd:', os.getcwd())

need = ['sobel_dma.bit', 'sobel_dma.hwh', 'sobel_dma.py', 'test_720p.png', 'sobel_720p.png']
missing = [f for f in need if not os.path.exists(f)]
print('missing:', missing if missing else 'none')
assert not missing, 'upload the missing files first'

In [ ]:
import sobel_dma
rc = sobel_dma.main(['--frames', '3', '--output', 'sobel_dma_out.png'])
print('return code:', rc)

## 结果怎么看

- `mismatched channels: 0` → 硬件和软件参考逐位一致，算子正确性成立。
- `throughput` 一行是这一版（按行传输）的帧率，**不是**最终实时性能；
  每次 DMA 传输的固定开销占了大头，真正的流式性能在 HDMI 版本里测。
- 输出图 `sobel_dma_out.png` 可以直接在 Jupyter 里打开看边缘效果。

如果比对失败：先把 `--frames` 调大（行缓冲需要预热），再确认参考图是用
`--border wrap` 生成的。